# Семинар: Продвинутые оптимизаторы градиентного спуска

**Цель семинара:** Реализовать и сравнить различные алгоритмы оптимизации, которые используются для обучения нейронных сетей и других моделей машинного обучения. 

В прошлом семинаре мы реализовали классический градиентный спуск. Однако у него есть недостатки:
1.  **Медленная сходимость** в областях с пологим градиентом.
2.  **Осцилляции** в областях с крутыми, вытянутыми "оврагами" функции потерь.
3.  **Риск застрять** в локальных минимумах или седловых точках.

Чтобы решить эти проблемы, были разработаны более продвинутые оптимизаторы. Сегодня мы реализуем и сравним самые популярные из них.

## 1. Подготовка: Функция потерь и её градиент

Для наглядности мы будем работать не с реальными данными, а с простой двумерной функцией, которая имитирует сложный ландшафт потерь. Это позволит нам красиво визуализировать пути спуска разных оптимизаторов.

Возьмем функцию, у которой есть вытянутый "овраг" — это настоящая проблема для простого градиентного спуска.
$$ f(w_0, w_1) = w_0^2 + 20w_1^2 $$
Минимум этой функции находится в точке (0, 0). Градиент по каждому параметру:
$$ \frac{\partial f}{\partial w_0} = 2w_0 $$
$$ \frac{\partial f}{\partial w_1} = 40w_1 $$

Весь основной код для запуска и визуализации уже написан. Ваша задача — реализовать **метод `update`** для каждого класса-оптимизатора.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Наша функция и ее градиент
def f(w):
    return w[0]**2 + 20 * w[1]**2

def grad_f(w):
    return np.array([2 * w[0], 40 * w[1]])

# ----- Вспомогательный код (уже написан для вас) -----

def run_optimizer(optimizer, start_w, num_iterations=50):
    """Запускает цикл оптимизации для заданного оптимизатора."""
    path = [start_w.copy()]
    w = start_w.copy()
    
    for _ in range(num_iterations):
        grads = grad_f(w)
        w = optimizer.update(w, grads)
        path.append(w.copy())
        
    return np.array(path)

def plot_paths_and_loss(histories, title):
    """Рисует контурный график и кривые потерь."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Контурный график
    w0_vals = np.linspace(-10, 10, 100)
    w1_vals = np.linspace(-3, 3, 100)
    W0, W1 = np.meshgrid(w0_vals, w1_vals)
    Z = f([W0, W1])
    
    ax1.contour(W0, W1, Z, levels=np.logspace(0, 3.5, 10), cmap='viridis')
    ax1.set_title('Пути оптимизаторов на ландшафте потерь')
    ax1.set_xlabel('$w_0$')
    ax1.set_ylabel('$w_1$')
    ax1.axhline(0, color='gray', linestyle='--')
    ax1.axvline(0, color='gray', linestyle='--')

    # График потерь
    ax2.set_title('Значение функции потерь (log scale)')
    ax2.set_xlabel('Итерация')
    ax2.set_ylabel('Loss (f(w))')
    ax2.set_yscale('log')
    
    for name, path in histories.items():
        ax1.plot(path[:, 0], path[:, 1], '-o', label=name, markersize=3)
        
        loss_values = [f(w) for w in path]
        ax2.plot(loss_values, label=name)
        
    ax1.legend()
    ax2.legend()
    fig.suptitle(title, fontsize=16)
    plt.show()

## Задание 1: Стохастический градиентный спуск (SGD)

Это базовый алгоритм, который мы уже знаем. Он делает шаг в сторону антиградиента. Мы будем использовать его как точку отсчета.

**Формула:**
$$ w_{t+1} = w_t - \alpha \cdot \nabla f(w_t) $$

**Ваша задача:** Реализовать метод `update` в классе `SGD`.

In [ ]:
class SGD:
    def __init__(self, learning_rate=0.04):
        self.lr = learning_rate
        
    def update(self, weights, grads):
        # TODO: Реализуйте обновление весов для SGD
        new_weights = ...
        
        return new_weights


## Задание 2: Оптимизатор с моментом (Momentum)

Momentum помогает ускорить спуск в правильном направлении и сгладить осцилляции. Представьте себе шарик, который катится с горы. Он набирает инерцию и не так сильно реагирует на мелкие неровности ландшафта. 

Вводится вектор "скорости" $v$, который является экспоненциально взвешенным средним предыдущих градиентов.

**Формулы:**
$$ v_t = \beta v_{t-1} + \alpha \nabla f(w_t) $$
$$ w_{t+1} = w_t - v_t $$
где $\beta$ – коэффициент инерции (обычно ~0.9).

**Ваша задача:** Реализовать метод `update` в классе `Momentum`. Не забудьте инициализировать и обновлять скорость `self.velocity`.

In [ ]:
class Momentum:
    def __init__(self, learning_rate=0.04, beta=0.9):
        self.lr = learning_rate
        self.beta = beta
        self.velocity = 0 # Начальная скорость равна нулю
        
    def update(self, weights, grads):
        # TODO: Реализуйте обновление скорости и весов
        # 1. Обновить self.velocity
        self.velocity = ...
        # 2. Обновить веса, используя новую скорость
        new_weights = ...
        
        return new_weights


## Задание 3: RMSProp (Root Mean Square Propagation)

RMSProp вводит адаптивную скорость обучения для каждого параметра. Идея в том, чтобы для параметров, по которым градиент большой и частый (сильные осцилляции), уменьшать скорость обучения, а для параметров с маленьким градиентом — увеличивать.

Для этого он хранит скользящее среднее квадратов градиентов $S_t$.

**Формулы:**
$$ S_t = \beta S_{t-1} + (1 - \beta)(\nabla f(w_t))^2 $$
$$ w_{t+1} = w_t - \alpha \frac{\nabla f(w_t)}{\sqrt{S_t} + \epsilon} $$

- $\beta$ – коэффициент затухания (обычно 0.999).
- $\epsilon$ – малая константа для численной стабильности (например, $10^{-8}$).

**Ваша задача:** Реализовать метод `update` для `RMSProp`.

In [ ]:
class RMSProp:
    def __init__(self, learning_rate=0.4, beta=0.99, epsilon=1e-8):
        self.lr = learning_rate
        self.beta = beta
        self.epsilon = epsilon
        self.s = 0 # Скользящее среднее квадратов градиентов
        
    def update(self, weights, grads):
        # TODO: Реализуйте обновление s и весов
        # 1. Обновить self.s
        self.s = ...
        # 2. Обновить веса
        new_weights = ...
        
        return new_weights

## Задание 4: Adam (Adaptive Moment Estimation)

Adam — король оптимизаторов. Он комбинирует идеи Momentum (инерция) и RMSProp (адаптивная скорость обучения).

- Он хранит скользящее среднее градиентов $m_t$ (как Momentum).
- Он хранит скользящее среднее квадратов градиентов $v_t$ (как RMSProp).
- Он делает коррекцию смещения для $m_t$ и $v_t$, так как вначале они смещены к нулю.

**Формулы:**
1. Обновление моментов:
$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla f(w_t) $$ 
$$ v_t = \beta_2 v_{t-1} + (1 - \beta_2) (\nabla f(w_t))^2 $$ 
2. Коррекция смещения:
$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t} $$ 
$$ \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $$ 
3. Обновление весов:
$$ w_{t+1} = w_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} $$ 

**Ваша задача:** Реализовать метод `update` для `Adam`. Вам понадобится счетчик шагов `self.t`.

In [ ]:
class Adam:
    def __init__(self, learning_rate=0.5, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = 0 # Первый момент (как velocity в momentum)
        self.v = 0 # Второй момент (как s в RMSProp)
        self.t = 0 # Счетчик шагов 
        
    def update(self, weights, grads):
        # Увеличиваем счетчик шагов
        self.t += 1
        
        # TODO: Реализуйте полный шаг Adam
        # 1. Обновить self.m и self.v
        self.m = ...
        self.v = ...
        
        # 2. Сделать коррекцию смещения
        m_hat = ...
        v_hat = ... 
        
        # 3. Обновить веса
        new_weights = ...
        
        return new_weights


## 5. Сравнение оптимизаторов

Теперь, когда все оптимизаторы реализованы, запустим их из одной и той же стартовой точки и посмотрим, кто быстрее и лучше справится с задачей.

**Для корректной работы этого блока убедитесь, что вы заполнили код во всех предыдущих заданиях.** Просто запустите ячейку ниже.

In [ ]:
# Стартовая точка для всех
start_w = np.array([9.5, 2.5])
iterations = 70

# Инициализация оптимизаторов
optimizers = {
    'SGD': SGD(),
    'Momentum': Momentum(),
    'RMSProp': RMSProp(),
    'Adam': Adam()
}

histories = {}

# Запускаем и сохраняем истории
for name, optimizer in optimizers.items():
    try:
        path = run_optimizer(optimizer, start_w, num_iterations=iterations)
        histories[name] = path
        print(f'Оптимизатор {name} успешно запущен.')
    except (TypeError, NotImplementedError):
        print(f'ОШИБКА: Оптимизатор {name} не реализован или реализован неверно.')
        
# Визуализация
if histories:
    plot_paths_and_loss(histories, 'Сравнение оптимизаторов')


## Выводы

Посмотрите на графики. Вы должны увидеть примерно следующее:
- **SGD** движется очень медленно по оси $w_1$ и сильно осциллирует по оси $w_0$.
- **Momentum** за счет инерции "пролетает" по оврагу гораздо быстрее и с меньшими колебаниями.
- **RMSProp** адаптирует шаг: он делает большие шаги по "пологой" оси $w_1$ и маленькие по "крутой" оси $w_0$, двигаясь к цели почти по прямой.
- **Adam** сочетает преимущества обоих и, как правило, сходится быстрее и стабильнее всех.

Поздравляем! Вы реализовали ключевые алгоритмы, которые лежат в основе обучения современных нейронных сетей.